# Data Compilation & Smoothing

1. **Data Compilation** — build master trajectory CSVs from the raw per-trajectory files.
2. **Trajectory Smoothing (Savgol Filter)** — smooth the master files (produces the `_smoothed.csv` files).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Data Compilation

Builds a master trajectory CSV for each dataset from the raw per-trajectory files.

In [ ]:
## Compiling Vision Data 1
import pandas as pd
import numpy as np
import glob
import os
import re

# 1. Define the path to your CSV files
folder_path = '../csv_data/'
all_files = glob.glob(os.path.join(folder_path, "vision*.csv"))

merged_data = []

for file in all_files:
    filename = os.path.basename(file)
    
    # 2. Extract Bat and Trajectory numbers using Regex
    match = re.search(r'vision_?([A-Za-z]+|[0-9]+)_?(\d+)\.csv', filename, re.IGNORECASE)
    
    if match:
        bat_no = match.group(1)   
        traj_no = match.group(2)  
    else:
        bat_no = 'Unknown'
        traj_no = 'Unknown'
        
    # 3. Read the CSV file
    df = pd.read_csv(file)
    
    # --- FIX: Force coordinate columns to be numeric ---
    cols_to_convert = [
        'head1', 'head2', 'back', 
        'Var28', 'Var31', 'Var34', 
        'Var29', 'Var32', 'Var35',
        'Frame_', 'Time'
    ]
    
    for col in cols_to_convert:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    # 4. Calculate the centroids (ignoring NaNs automatically)
    df['pos_x'] = df[['head1', 'head2', 'back']].mean(axis=1)
    df['pos_y'] = df[['Var28', 'Var31', 'Var34']].mean(axis=1)
    df['pos_z'] = df[['Var29', 'Var32', 'Var35']].mean(axis=1)
    
    # 5. Extract only the required columns
    result_df = df[['Frame_', 'Time', 'pos_x', 'pos_y', 'pos_z']].copy()
    
    # Add the identifier columns
    result_df['bat no.'] = bat_no
    result_df['trajectory no.'] = traj_no
    
    # Drop rows where ALL position values are NaN 
    result_df = result_df.dropna(subset=['pos_x', 'pos_y', 'pos_z'], how='all')
    
    # --- NEW: Check if the dataframe is empty after dropping NaNs ---
    if result_df.empty:
        print(f"⚠️ Warning: '{filename}' resulted in empty data (Bat {bat_no}, Traj {traj_no}). Skipping.")
    else:
        merged_data.append(result_df)


# 6. Concatenate everything into one giant Dataframe
if merged_data:
    final_merged_df = pd.concat(merged_data, ignore_index=True)

    # Reorder columns slightly so the identifiers are at the front
    cols = ['bat no.', 'trajectory no.', 'Frame_', 'Time', 'pos_x', 'pos_y', 'pos_z']
    final_merged_df = final_merged_df[cols]

    # 7. Save to a new Master CSV
    final_merged_df.to_csv(os.path.join(folder_path, 'master_trajectory_data.csv'), index=False)

    print(f"\n✅ Successfully merged {len(merged_data)} valid files!")
    
    # ==============================================================
    # 8. NEW: Compare all trajectories for exact duplicates
    # ==============================================================
    print("\n--- Running Duplicate Trajectory Check ---")
    trajectories = {}
    
    # Group by both bat and trajectory number so we can easily compare them
    for (bat, traj), group in final_merged_df.groupby(['bat no.', 'trajectory no.']):
        # Store just the spatial coordinates as a numpy array for fast mathematical comparison
        trajectories[(bat, traj)] = group[['pos_x', 'pos_y', 'pos_z']].values

    duplicates_found = []
    keys = list(trajectories.keys())
    
    # Compare every trajectory to every other trajectory exactly once
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            key1, key2 = keys[i], keys[j]
            coords1, coords2 = trajectories[key1], trajectories[key2]

            # 1. Do they have the exact same number of recorded points?
            if coords1.shape == coords2.shape:
                # 2. Are all values mathematically identical? (equal_nan=True allows NaNs to match)
                if np.array_equal(coords1, coords2, equal_nan=True):
                    duplicates_found.append((key1, key2))

    if duplicates_found:
        print(f"🚨 Found {len(duplicates_found)} pair(s) of EXACT duplicate physical trajectories:")
        for pair in duplicates_found:
            print(f"   -> Bat {pair[0][0]} Traj {pair[0][1]} is a perfect clone of Bat {pair[1][0]} Traj {pair[1][1]}")
    else:
        print("✅ No exact duplicate trajectories found in the dataset.")

else:
    print("No data was merged. Please check your folder path.")

In [ ]:
#Compiling Acoustic Data of 24Dec
import pandas as pd
import numpy as np
import glob
import os
import re

# ============================================================
# 1. Locate all CSV files
# ============================================================

folder_path = "../csv_acoustic/csv_24Dec24"

all_files = glob.glob(os.path.join(folder_path, "*.csv"))

merged_data = []

print(f"Found {len(all_files)} csv files.\n")

# ============================================================
# 2. Process each file
# ============================================================

for file in all_files:

    filename = os.path.basename(file)

    # --------------------------------------------------------
    # Extract bat name and trajectory number
    #
    # Matches:
    # pitt1edited
    # pitt1
    # pitt27edited
    # pitt27
    # --------------------------------------------------------

    match = re.search(
        r'_([A-Za-z]+)(\d+)(?:edited)?_.*?_mat\.csv$',
        filename,
        re.IGNORECASE
    )

    if match:
        bat_name = match.group(1)
        traj_no = match.group(2)
    else:
        bat_name = "Unknown"
        traj_no = "Unknown"

    try:

        # ----------------------------------------------------
        # Read CSV with double header
        # ----------------------------------------------------

        df = pd.read_csv(file, header=[0, 1])

        # Keep only first header level
        df.columns = df.columns.get_level_values(0)

        # ----------------------------------------------------
        # Convert coordinates to numeric
        # ----------------------------------------------------

        cols_to_convert = [
            'Frame_',
            'Time',
            'head1',
            'head2',
            'back',
            'Var28',
            'Var31',
            'Var34',
            'Var29',
            'Var32',
            'Var35'
        ]

        for col in cols_to_convert:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # ----------------------------------------------------
        # Verify required columns exist
        # ----------------------------------------------------

        required_cols = [
            'head1', 'head2', 'back',
            'Var28', 'Var31', 'Var34',
            'Var29', 'Var32', 'Var35'
        ]

        missing = [c for c in required_cols if c not in df.columns]

        if missing:
            print(f"⚠️ {filename}")
            print(f"   Missing columns: {missing}")
            continue

        # ----------------------------------------------------
        # Calculate centroid position
        # ----------------------------------------------------

        df['pos_x'] = df[['head1', 'head2', 'back']].mean(axis=1)

        df['pos_y'] = df[['Var28', 'Var31', 'Var34']].mean(axis=1)

        df['pos_z'] = df[['Var29', 'Var32', 'Var35']].mean(axis=1)

        # ----------------------------------------------------
        # Keep only required columns
        # ----------------------------------------------------

        result_df = df[
            ['Frame_', 'Time', 'pos_x', 'pos_y', 'pos_z']
        ].copy()

        result_df['bat_name'] = bat_name
        result_df['trajectory_no'] = traj_no

        # ----------------------------------------------------
        # Remove rows with no coordinates
        # ----------------------------------------------------

        result_df = result_df.dropna(
            subset=['pos_x', 'pos_y', 'pos_z'],
            how='all'
        )

        if result_df.empty:

            print(
                f"⚠️ '{filename}' "
                f"(Bat={bat_name}, Traj={traj_no}) "
                f"contains no usable coordinates."
            )

        else:

            merged_data.append(result_df)

    except Exception as e:

        print(f"❌ Error processing {filename}")
        print(e)

# ============================================================
# 3. Combine all trajectories
# ============================================================

if merged_data:

    final_merged_df = pd.concat(
        merged_data,
        ignore_index=True
    )

    cols = [
        'bat_name',
        'trajectory_no',
        'Frame_',
        'Time',
        'pos_x',
        'pos_y',
        'pos_z'
    ]

    final_merged_df = final_merged_df[cols]

    # --------------------------------------------------------
    # Save master file
    # --------------------------------------------------------

    output_file = os.path.join(
        folder_path,
        "master_trajectory_data.csv"
    )

    final_merged_df.to_csv(
        output_file,
        index=False
    )

    print("\n===================================================")
    print(f"✅ Successfully merged {len(merged_data)} files")
    print(f"✅ Saved to: {output_file}")
    print("===================================================\n")

    # =======================================================
    # 4. Duplicate trajectory check
    # =======================================================

    print("Running duplicate trajectory check...\n")

    trajectories = {}

    for (bat, traj), group in final_merged_df.groupby(
        ['bat_name', 'trajectory_no']
    ):

        trajectories[(bat, traj)] = group[
            ['pos_x', 'pos_y', 'pos_z']
        ].values

    duplicates_found = []

    keys = list(trajectories.keys())

    for i in range(len(keys)):

        for j in range(i + 1, len(keys)):

            key1 = keys[i]
            key2 = keys[j]

            coords1 = trajectories[key1]
            coords2 = trajectories[key2]

            if coords1.shape == coords2.shape:

                if np.array_equal(
                    coords1,
                    coords2,
                    equal_nan=True
                ):

                    duplicates_found.append(
                        (key1, key2)
                    )

    if duplicates_found:

        print(
            f"🚨 Found {len(duplicates_found)} exact duplicate trajectory pairs:\n"
        )

        for pair in duplicates_found:

            print(
                f"Bat {pair[0][0]} Traj {pair[0][1]}"
                f"  <==>  "
                f"Bat {pair[1][0]} Traj {pair[1][1]}"
            )

    else:

        print("✅ No exact duplicate trajectories found.")

else:

    print("❌ No valid trajectory data found.")

In [ ]:
#Compiling Acoustic Data of 26Dec
import pandas as pd
import numpy as np
import glob
import os
import re

# ============================================================
# 1. Locate all CSV files
# ============================================================

folder_path = "../csv_acoustic/csv_26DDec24"

all_files = glob.glob(os.path.join(folder_path, "*.csv"))

merged_data = []

print(f"Found {len(all_files)} csv files.\n")

# ============================================================
# 2. Process each file
# ============================================================

for file in all_files:

    filename = os.path.basename(file)


    match = re.search(
        r'^[A-Z]{2}([A-Za-z]+)(\d+)\.csv$',
        filename,
        re.IGNORECASE
    )

    if match:
        bat_name = match.group(1)
        traj_no = match.group(2)
    else:
        bat_name = "Unknown"
        traj_no = "Unknown"

    try:

        # ----------------------------------------------------
        # Read CSV with double header
        # ----------------------------------------------------

        df = pd.read_csv(file, header=[0, 1])

        # Keep only first header level
        df.columns = df.columns.get_level_values(0)

        # ----------------------------------------------------
        # Convert coordinates to numeric
        # ----------------------------------------------------

        cols_to_convert = [
            'Frame_',
            'Time',
            'head1',
            'head2',
            'back',
            'Var28',
            'Var31',
            'Var34',
            'Var29',
            'Var32',
            'Var35'
        ]

        for col in cols_to_convert:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # ----------------------------------------------------
        # Verify required columns exist
        # ----------------------------------------------------

        required_cols = [
            'head1', 'head2', 'back',
            'Var28', 'Var31', 'Var34',
            'Var29', 'Var32', 'Var35'
        ]

        missing = [c for c in required_cols if c not in df.columns]

        if missing:
            print(f"⚠️ {filename}")
            print(f"   Missing columns: {missing}")
            continue

        # ----------------------------------------------------
        # Calculate centroid position
        # ----------------------------------------------------

        df['pos_x'] = df[['head1', 'head2', 'back']].mean(axis=1)

        df['pos_y'] = df[['Var28', 'Var31', 'Var34']].mean(axis=1)

        df['pos_z'] = df[['Var29', 'Var32', 'Var35']].mean(axis=1)

        # ----------------------------------------------------
        # Keep only required columns
        # ----------------------------------------------------

        result_df = df[
            ['Frame_', 'Time', 'pos_x', 'pos_y', 'pos_z']
        ].copy()

        result_df['bat_name'] = bat_name
        result_df['trajectory_no'] = traj_no

        # ----------------------------------------------------
        # Remove rows with no coordinates
        # ----------------------------------------------------

        result_df = result_df.dropna(
            subset=['pos_x', 'pos_y', 'pos_z'],
            how='all'
        )

        if result_df.empty:

            print(
                f"⚠️ '{filename}' "
                f"(Bat={bat_name}, Traj={traj_no}) "
                f"contains no usable coordinates."
            )

        else:

            merged_data.append(result_df)

    except Exception as e:

        print(f"❌ Error processing {filename}")
        print(e)

# ============================================================
# 3. Combine all trajectories
# ============================================================

if merged_data:

    final_merged_df = pd.concat(
        merged_data,
        ignore_index=True
    )

    cols = [
        'bat_name',
        'trajectory_no',
        'Frame_',
        'Time',
        'pos_x',
        'pos_y',
        'pos_z'
    ]

    final_merged_df = final_merged_df[cols]

    # --------------------------------------------------------
    # Save master file
    # --------------------------------------------------------

    output_file = os.path.join(
        folder_path,
        "master_trajectory_data.csv"
    )

    final_merged_df.to_csv(
        output_file,
        index=False
    )

    print("\n===================================================")
    print(f"✅ Successfully merged {len(merged_data)} files")
    print(f"✅ Saved to: {output_file}")
    print("===================================================\n")

    # =======================================================
    # 4. Duplicate trajectory check
    # =======================================================

    print("Running duplicate trajectory check...\n")

    trajectories = {}

    for (bat, traj), group in final_merged_df.groupby(
        ['bat_name', 'trajectory_no']
    ):

        trajectories[(bat, traj)] = group[
            ['pos_x', 'pos_y', 'pos_z']
        ].values

    duplicates_found = []

    keys = list(trajectories.keys())

    for i in range(len(keys)):

        for j in range(i + 1, len(keys)):

            key1 = keys[i]
            key2 = keys[j]

            coords1 = trajectories[key1]
            coords2 = trajectories[key2]

            if coords1.shape == coords2.shape:

                if np.array_equal(
                    coords1,
                    coords2,
                    equal_nan=True
                ):

                    duplicates_found.append(
                        (key1, key2)
                    )

    if duplicates_found:

        print(
            f"🚨 Found {len(duplicates_found)} exact duplicate trajectory pairs:\n"
        )

        for pair in duplicates_found:

            print(
                f"Bat {pair[0][0]} Traj {pair[0][1]}"
                f"  <==>  "
                f"Bat {pair[1][0]} Traj {pair[1][1]}"
            )

    else:

        print("✅ No exact duplicate trajectories found.")

else:

    print("❌ No valid trajectory data found.")

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re

# ============================================================
# Compiling Acoustic Data of 30Oct24
# ============================================================

folder_path = "../csv_acoustic/csv_30Oct24"

all_files = glob.glob(os.path.join(folder_path, "*.csv"))

print(f"Found {len(all_files)} csv files.\n")

trajectory_store = {}


# ============================================================
# Helper function
# ============================================================

def canonical_name(filename):
    """
    Makes original and _Copy_ versions map
    to the same trajectory key.
    """

    return filename.replace("_Copy_", "_")


# ============================================================
# Process files
# ============================================================

for file in all_files:

    filename = os.path.basename(file)

    # --------------------------------------------------------
    # Extract bat and trajectory number
    # --------------------------------------------------------

    match = re.search(
        r'_(brad|ketem|pitt)(\d+)(?:edited)?',
        filename,
        re.IGNORECASE
    )

    if match:
        bat_name = match.group(1).lower()
        traj_no = int(match.group(2))
    else:
        bat_name = "Unknown"
        traj_no = -1

    try:

        # ----------------------------------------------------
        # Read csv
        # ----------------------------------------------------

        df = pd.read_csv(file, header=[0, 1])

        # Flatten multi-index header
        df.columns = df.columns.get_level_values(0)

        # ----------------------------------------------------
        # Convert numeric columns
        # ----------------------------------------------------

        cols_to_convert = [
            'Frame_',
            'Time',
            'head1',
            'head2',
            'back',
            'Var28',
            'Var31',
            'Var34',
            'Var29',
            'Var32',
            'Var35'
        ]

        for col in cols_to_convert:

            if col in df.columns:
                df[col] = pd.to_numeric(
                    df[col],
                    errors='coerce'
                )

        # ----------------------------------------------------
        # Verify required columns
        # ----------------------------------------------------

        required_cols = [
            'head1',
            'head2',
            'back',
            'Var28',
            'Var31',
            'Var34',
            'Var29',
            'Var32',
            'Var35'
        ]

        missing = [
            c for c in required_cols
            if c not in df.columns
        ]

        if missing:

            print(f"⚠️ {filename}")
            print(f"   Missing columns: {missing}")

            continue

        # ----------------------------------------------------
        # Compute centroid
        # ----------------------------------------------------

        df['pos_x'] = (
            df[['head1', 'head2', 'back']]
            .mean(axis=1)
        )

        df['pos_y'] = (
            df[['Var28', 'Var31', 'Var34']]
            .mean(axis=1)
        )

        df['pos_z'] = (
            df[['Var29', 'Var32', 'Var35']]
            .mean(axis=1)
        )

        # ----------------------------------------------------
        # Keep required columns
        # ----------------------------------------------------

        result_df = df[
            [
                'Frame_',
                'Time',
                'pos_x',
                'pos_y',
                'pos_z'
            ]
        ].copy()

        result_df['bat_name'] = bat_name
        result_df['trajectory_no'] = traj_no

        # ----------------------------------------------------
        # Remove empty rows
        # ----------------------------------------------------

        result_df = result_df.dropna(
            subset=['pos_x', 'pos_y', 'pos_z'],
            how='all'
        )

        if result_df.empty:

            print(
                f"⚠️ '{filename}' "
                f"(Bat={bat_name}, Traj={traj_no}) "
                f"contains no usable coordinates."
            )

            continue

        # ----------------------------------------------------
        # Handle Copy files
        # ----------------------------------------------------

        key = canonical_name(filename)

        if key not in trajectory_store:

            trajectory_store[key] = {
                "filename": filename,
                "data": result_df
            }

        else:

            old_df = trajectory_store[key]["data"]

            old_coords = old_df[
                ['pos_x', 'pos_y', 'pos_z']
            ].values

            new_coords = result_df[
                ['pos_x', 'pos_y', 'pos_z']
            ].values

            identical = (
                old_coords.shape == new_coords.shape
                and np.array_equal(
                    old_coords,
                    new_coords,
                    equal_nan=True
                )
            )

            if identical:

                print(
                    f"\nCopy identical to original:"
                    f"\n   Keeping: {trajectory_store[key]['filename']}"
                    f"\n   Ignoring: {filename}\n"
                )

            else:

                old_valid = (
                    old_df[
                        ['pos_x', 'pos_y', 'pos_z']
                    ]
                    .notna()
                    .all(axis=1)
                    .sum()
                )

                new_valid = (
                    result_df[
                        ['pos_x', 'pos_y', 'pos_z']
                    ]
                    .notna()
                    .all(axis=1)
                    .sum()
                )

                print(
                    f"\nWARNING: Copy differs from original"
                    f"\n   Original: {trajectory_store[key]['filename']}"
                    f"\n   Copy:     {filename}"
                    f"\n   Valid points: {old_valid} vs {new_valid}"
                )

                if new_valid > old_valid:

                    print(
                        f"   -> Keeping copy\n"
                    )

                    trajectory_store[key] = {
                        "filename": filename,
                        "data": result_df
                    }

                else:

                    print(
                        f"   -> Keeping original\n"
                    )

    except Exception as e:

        print(
            f"\n❌ Error processing {filename}"
        )

        print(e)

# ============================================================
# Build final merged dataset
# ============================================================

merged_data = [
    item["data"]
    for item in trajectory_store.values()
]

if merged_data:

    final_merged_df = pd.concat(
        merged_data,
        ignore_index=True
    )

    final_merged_df = final_merged_df[
        [
            'bat_name',
            'trajectory_no',
            'Frame_',
            'Time',
            'pos_x',
            'pos_y',
            'pos_z'
        ]
    ]

    # --------------------------------------------------------
    # Save master csv
    # --------------------------------------------------------

    output_file = os.path.join(
        folder_path,
        "master_trajectory_data.csv"
    )

    final_merged_df.to_csv(
        output_file,
        index=False
    )

    print("\n===================================================")
    print(
        f"✅ Successfully merged "
        f"{len(merged_data)} trajectories"
    )
    print(f"✅ Saved to: {output_file}")
    print("===================================================\n")

    # =======================================================
    # Dataset summary
    # =======================================================

    print("Trajectory counts by bat:\n")

    summary = (
        final_merged_df
        .groupby('bat_name')['trajectory_no']
        .nunique()
        .sort_index()
    )

    for bat, count in summary.items():

        print(
            f"{bat:<10} : {count}"
        )

    # =======================================================
    # Duplicate trajectory check
    # =======================================================

    print("\nRunning duplicate trajectory check...\n")

    trajectories = {}

    for (bat, traj), group in final_merged_df.groupby(
        ['bat_name', 'trajectory_no']
    ):

        trajectories[(bat, traj)] = group[
            ['pos_x', 'pos_y', 'pos_z']
        ].values

    duplicates_found = []

    keys = list(trajectories.keys())

    for i in range(len(keys)):

        for j in range(i + 1, len(keys)):

            coords1 = trajectories[keys[i]]
            coords2 = trajectories[keys[j]]

            if (
                coords1.shape == coords2.shape
                and np.array_equal(
                    coords1,
                    coords2,
                    equal_nan=True
                )
            ):

                duplicates_found.append(
                    (keys[i], keys[j])
                )

    if duplicates_found:

        print(
            f"🚨 Found "
            f"{len(duplicates_found)} duplicate "
            f"trajectory pair(s):\n"
        )

        for pair in duplicates_found:

            print(
                f"{pair[0][0]} traj {pair[0][1]}"
                f"  <==>  "
                f"{pair[1][0]} traj {pair[1][1]}"
            )

    else:

        print(
            "✅ No duplicate physical trajectories found."
        )

else:

    print(
        "❌ No valid trajectory data found."
    )

In [ ]:
# ============================================================
# Compiling Acoustic Data of 31Dec24
#
# Features:
# 1. Reads all trajectories
# 2. Detects original vs edited versions
# 3. Compares original and edited trajectories
# 4. Reports differences
# 5. Always keeps the edited version if available
# 6. Creates master_trajectory_data.csv
# 7. Checks for duplicate physical trajectories
# ============================================================

import pandas as pd
import numpy as np
import glob
import os
import re

# ============================================================
# SETTINGS
# ============================================================

folder_path = "../csv_acoustic/csv_31Dec24"

all_files = glob.glob(os.path.join(folder_path, "*.csv"))

print(f"Found {len(all_files)} csv files.\n")

# ============================================================
# STORAGE
# ============================================================

trajectory_versions = {}


# ============================================================
# PROCESS FILES
# ============================================================

for file in all_files:

    filename = os.path.basename(file)

    # --------------------------------------------------------
    # Extract:
    # ketem6
    # ketem6edited
    # --------------------------------------------------------

    match = re.search(
        r'_(ketem)(\d+)(edited)?_',
        filename,
        re.IGNORECASE
    )

    if match:

        bat_name = match.group(1).lower()
        traj_no = int(match.group(2))
        is_edited = match.group(3) is not None

    else:

        bat_name = "Unknown"
        traj_no = -1
        is_edited = False

    try:

        # ----------------------------------------------------
        # Read csv
        # ----------------------------------------------------

        df = pd.read_csv(file, header=[0, 1])

        # Flatten MultiIndex columns
        df.columns = df.columns.get_level_values(0)

        # ----------------------------------------------------
        # Convert numeric columns
        # ----------------------------------------------------

        cols_to_convert = [
            'Frame_',
            'Time',
            'head1',
            'head2',
            'back',
            'Var28',
            'Var31',
            'Var34',
            'Var29',
            'Var32',
            'Var35'
        ]

        for col in cols_to_convert:

            if col in df.columns:

                df[col] = pd.to_numeric(
                    df[col],
                    errors='coerce'
                )

        # ----------------------------------------------------
        # Verify required columns exist
        # ----------------------------------------------------

        required_cols = [
            'head1',
            'head2',
            'back',
            'Var28',
            'Var31',
            'Var34',
            'Var29',
            'Var32',
            'Var35'
        ]

        missing = [
            c for c in required_cols
            if c not in df.columns
        ]

        if missing:

            print(f"⚠️ {filename}")
            print(f"Missing columns: {missing}")

            continue

        # ----------------------------------------------------
        # Compute centroid
        # ----------------------------------------------------

        df['pos_x'] = (
            df[['head1', 'head2', 'back']]
            .mean(axis=1)
        )

        df['pos_y'] = (
            df[['Var28', 'Var31', 'Var34']]
            .mean(axis=1)
        )

        df['pos_z'] = (
            df[['Var29', 'Var32', 'Var35']]
            .mean(axis=1)
        )

        # ----------------------------------------------------
        # Keep required columns
        # ----------------------------------------------------

        result_df = df[
            [
                'Frame_',
                'Time',
                'pos_x',
                'pos_y',
                'pos_z'
            ]
        ].copy()

        result_df['bat_name'] = bat_name
        result_df['trajectory_no'] = traj_no

        # ----------------------------------------------------
        # Remove empty rows
        # ----------------------------------------------------

        result_df = result_df.dropna(
            subset=['pos_x', 'pos_y', 'pos_z'],
            how='all'
        )

        if result_df.empty:

            print(
                f"⚠️ {filename} contains no valid coordinates."
            )

            continue

        # ----------------------------------------------------
        # Store trajectory version
        # ----------------------------------------------------

        key = (bat_name, traj_no)

        if key not in trajectory_versions:

            trajectory_versions[key] = {}

        if is_edited:

            trajectory_versions[key]["edited"] = {
                "filename": filename,
                "data": result_df
            }

        else:

            trajectory_versions[key]["original"] = {
                "filename": filename,
                "data": result_df
            }

    except Exception as e:

        print(f"\n❌ Error processing {filename}")
        print(e)

# ============================================================
# COMPARE EDITED VS ORIGINAL
# ============================================================

print("\n")
print("=" * 60)
print("EDITED vs ORIGINAL COMPARISON")
print("=" * 60)

merged_data = []

for key in sorted(trajectory_versions.keys()):

    versions = trajectory_versions[key]

    edited = versions.get("edited")
    original = versions.get("original")

    # --------------------------------------------------------
    # BOTH EXIST
    # --------------------------------------------------------

    if edited is not None and original is not None:

        edited_df = edited["data"]
        original_df = original["data"]

        e = edited_df[
            ['pos_x', 'pos_y', 'pos_z']
        ].values

        o = original_df[
            ['pos_x', 'pos_y', 'pos_z']
        ].values

        print(
            f"\nTrajectory {key[1]}"
        )

        identical = (
            e.shape == o.shape
            and np.array_equal(
                e,
                o,
                equal_nan=True
            )
        )

        if identical:

            print("  ✅ Edited and original are IDENTICAL")

        else:

            print("  ⚠️ Edited and original differ")

            if e.shape == o.shape:

                diff_mask = ~np.isclose(
                    e,
                    o,
                    equal_nan=True
                )

                differing_frames = np.any(
                    diff_mask,
                    axis=1
                ).sum()

                try:

                    max_difference = np.nanmax(
                        np.abs(e - o)
                    )

                except ValueError:

                    max_difference = np.nan

                print(
                    f"     Differing frames : "
                    f"{differing_frames}"
                )

                print(
                    f"     Max coordinate difference : "
                    f"{max_difference:.4f}"
                )

            else:

                print(
                    f"     Different lengths:"
                    f" {len(o)} vs {len(e)}"
                )

        print(
            f"  → Using edited version"
        )

        merged_data.append(edited_df)

    # --------------------------------------------------------
    # ONLY EDITED EXISTS
    # --------------------------------------------------------

    elif edited is not None:

        print(
            f"\nTrajectory {key[1]}"
            f" : edited only"
        )

        merged_data.append(
            edited["data"]
        )

    # --------------------------------------------------------
    # ONLY ORIGINAL EXISTS
    # --------------------------------------------------------

    elif original is not None:

        print(
            f"\nTrajectory {key[1]}"
            f" : original only"
        )

        merged_data.append(
            original["data"]
        )

# ============================================================
# BUILD MASTER DATASET
# ============================================================

if merged_data:

    final_merged_df = pd.concat(
        merged_data,
        ignore_index=True
    )

    final_merged_df = final_merged_df[
        [
            'bat_name',
            'trajectory_no',
            'Frame_',
            'Time',
            'pos_x',
            'pos_y',
            'pos_z'
        ]
    ]

    # --------------------------------------------------------
    # Save master csv
    # --------------------------------------------------------

    output_file = os.path.join(
        folder_path,
        "master_trajectory_data.csv"
    )

    final_merged_df.to_csv(
        output_file,
        index=False
    )

    print("\n")
    print("=" * 60)
    print("MASTER FILE CREATED")
    print("=" * 60)

    print(
        f"Saved to:\n{output_file}"
    )

    print(
        f"\nTotal trajectories used: "
        f"{len(merged_data)}"
    )

    # =======================================================
    # DATASET SUMMARY
    # =======================================================

    print("\nTrajectory counts:\n")

    summary = (
        final_merged_df
        .groupby("bat_name")["trajectory_no"]
        .nunique()
        .sort_index()
    )

    for bat, count in summary.items():

        print(
            f"{bat:<10} : {count}"
        )

    # =======================================================
    # DUPLICATE TRAJECTORY CHECK
    # =======================================================

    print("\n")
    print("=" * 60)
    print("DUPLICATE TRAJECTORY CHECK")
    print("=" * 60)

    trajectories = {}

    for (bat, traj), group in final_merged_df.groupby(
        ['bat_name', 'trajectory_no']
    ):

        trajectories[(bat, traj)] = (
            group[
                ['pos_x', 'pos_y', 'pos_z']
            ]
            .values
        )

    keys = list(
        trajectories.keys()
    )

    duplicates_found = []

    for i in range(len(keys)):

        for j in range(i + 1, len(keys)):

            coords1 = trajectories[keys[i]]
            coords2 = trajectories[keys[j]]

            if (
                coords1.shape == coords2.shape
                and np.array_equal(
                    coords1,
                    coords2,
                    equal_nan=True
                )
            ):

                duplicates_found.append(
                    (keys[i], keys[j])
                )

    if duplicates_found:

        print(
            f"\n🚨 Found "
            f"{len(duplicates_found)} "
            f"duplicate trajectory pair(s):\n"
        )

        for pair in duplicates_found:

            print(
                f"{pair[0][0]} traj {pair[0][1]}"
                f"  <==>  "
                f"{pair[1][0]} traj {pair[1][1]}"
            )

    else:

        print(
            "\n✅ No duplicate physical trajectories found."
        )

else:

    print(
        "\n❌ No valid trajectory data found."
    )

In [12]:

# Robust compiler for 1 Oct 2022 trajectories
#
# Features
# --------
# • Automatically discovers filename variants
# • Priority:
#     Smoothed+NoRec > Smoothed > NoRec > Original
# • Handles typos such as:
#     SmoothingNoRec
#     smoothed
#     smoothes
#     NoRec2MarSmoothed
# • Ignores "sketem"
# • Works for both 50-column and 101-column csv files
# • Checks required columns
# • Removes empty trajectories
# • Detects duplicate trajectories
# • Writes master csv

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------

DATA_DIR = Path("../csv_vision/csv_1Oct22")
OUTPUT_FILE = DATA_DIR / "master_acoustic_trajectory_data_1Oct22.csv"

DATE_TOKEN = "01oct"

# ------------------------------------------------------------------

REQUIRED = [
    "head1","head2","back",
    "Var28","Var29",
    "Var31","Var32",
    "Var34","Var35"
]

PATTERN = re.compile(
    r'([A-Za-z]+)' + DATE_TOKEN + r'(\d+)',
    re.IGNORECASE
)


def classify(filename):

    n = filename.lower()

    smooth = "smooth" in n
    norec = "norec" in n

    if smooth and norec:
        return 0, "SmoothedNoRec"

    if smooth:
        return 1, "Smoothed"

    if norec:
        return 2, "NoRec"

    return 3, "Original"


def load_single_csv(path):

    try:
        df = pd.read_csv(path, header=[0,1])
    except Exception:
        df = pd.read_csv(path)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    missing = [c for c in REQUIRED if c not in df.columns]

    if missing:
        print(f"Missing columns in {path.name}")
        print(missing)
        return None

    numeric_cols = ["Frame_", "Time"] + REQUIRED

    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["pos_x"] = df[["head1","head2","back"]].mean(axis=1)
    df["pos_y"] = df[["Var28","Var31","Var34"]].mean(axis=1)
    df["pos_z"] = df[["Var29","Var32","Var35"]].mean(axis=1)

    keep = ["Frame_","Time","pos_x","pos_y","pos_z"]

    keep = [c for c in keep if c in df.columns]

    df = df[keep]

    df = df.dropna(
        subset=["pos_x","pos_y","pos_z"],
        how="all"
    )

    if len(df) == 0:
        return None

    return df


def main():

    files = sorted(DATA_DIR.glob("*.csv"))

    print(f"Found {len(files)} csv files")

    grouped = defaultdict(list)

    ignored = []

    for f in files:

        m = PATTERN.search(f.name)

        if m is None:
            print("Could not parse:", f.name)
            continue

        bat = m.group(1).lower()
        traj = int(m.group(2))

        if bat == "sketem":
            ignored.append(f.name)
            continue

        priority, label = classify(f.name)

        grouped[(bat, traj)].append(
            (priority, label, f)
        )

    merged = []

    print("\n" + "="*80)
    print("FILE SELECTION")
    print("="*80)

    for key in sorted(grouped):

        candidates = sorted(grouped[key], key=lambda x: x[0])

        print(f"\n{key}")

        for p,l,f in candidates:
            print(f"   [{p}] {l:14s} {f.name}")

        chosen = candidates[0]

        print("   --> USING:", chosen[2].name)

        df = load_single_csv(chosen[2])

        if df is None:
            print("      Empty / invalid.")
            continue

        df["bat_name"] = key[0]
        df["trajectory_no"] = key[1]
        df["source_file"] = chosen[2].name

        merged.append(df)

    if len(merged)==0:
        raise RuntimeError("No valid trajectories found.")

    master = pd.concat(
        merged,
        ignore_index=True
    )

    master = master[
        [
            "bat_name",
            "trajectory_no",
            "Frame_",
            "Time",
            "pos_x",
            "pos_y",
            "pos_z",
            "source_file"
        ]
    ]

    master.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print("\n")
    print("="*80)
    print("SUMMARY")
    print("="*80)

    print(master.groupby("bat_name")["trajectory_no"].nunique())

    print("\nIgnored files")
    for x in ignored:
        print(" ", x)

    print("\nChecking duplicate trajectories...")

    trajs = {}

    for key, g in master.groupby(["bat_name","trajectory_no"]):
        trajs[key] = g[["pos_x","pos_y","pos_z"]].to_numpy()

    keys = list(trajs)

    dup = False

    for i in range(len(keys)):
        for j in range(i+1, len(keys)):

            A = trajs[keys[i]]
            B = trajs[keys[j]]

            if A.shape != B.shape:
                continue

            if np.array_equal(A, B, equal_nan=True):
                print(keys[i], "<==>", keys[j])
                dup = True

    if not dup:
        print("No duplicate trajectories found.")

    print("\nSaved to")
    print(OUTPUT_FILE)


if __name__ == "__main__":
    main()


Found 89 csv files

FILE SELECTION

('ketem', 23)
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct23_1Oct22_mat.csv
   --> USING: D__data_mia_vision_3rec_1Oct22_ketem01oct23_1Oct22_mat.csv

('ketem', 24)
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct24_1Oct22_mat.csv
   --> USING: D__data_mia_vision_3rec_1Oct22_ketem01oct24_1Oct22_mat.csv

('ketem', 31)
   [0] SmoothedNoRec  D__data_mia_vision_3rec_1Oct22_ketem01oct31SmoothedNoRec_1Oct22_mat.csv
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct31_1Oct22_mat.csv
   --> USING: D__data_mia_vision_3rec_1Oct22_ketem01oct31SmoothedNoRec_1Oct22_mat.csv

('ketem', 56)
   [0] SmoothedNoRec  D__data_mia_vision_3rec_1Oct22_ketem01oct56SmoothedNoRec_1Oct22_mat.csv
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct56_1Oct22_mat.csv
   --> USING: D__data_mia_vision_3rec_1Oct22_ketem01oct56SmoothedNoRec_1Oct22_mat.csv

('ketem', 60)
   [0] SmoothedNoRec  D__data_mia_vision_3rec_1Oct22_ketem01

C:\Users\iswav\AppData\Local\Temp\ipykernel_26528\4281130745.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pos_x"] = df[["head1","head2","back"]].mean(axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_26528\4281130745.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pos_y"] = df[["Var28","Var31","Var34"]].mean(axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_26528\4281130745.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


('ketem', 78)
   [1] Smoothed       D__data_mia_vision_3rec_1Oct22_ketem01oct78smoothed_1Oct22_mat.csv
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct78_1Oct22_mat.csv
   --> USING: D__data_mia_vision_3rec_1Oct22_ketem01oct78smoothed_1Oct22_mat.csv

('ketem', 79)
   [1] Smoothed       D__data_mia_vision_3rec_1Oct22_ketem01oct79smoothed_1Oct22_mat.csv
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct79_1Oct22_mat.csv
   --> USING: D__data_mia_vision_3rec_1Oct22_ketem01oct79smoothed_1Oct22_mat.csv

('ketem', 80)
   [1] Smoothed       D__data_mia_vision_3rec_1Oct22_ketem01oct80smoothed_1Oct22_mat.csv
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct80_1Oct22_mat.csv
   --> USING: D__data_mia_vision_3rec_1Oct22_ketem01oct80smoothed_1Oct22_mat.csv

('ketem', 81)
   [0] SmoothedNoRec  D__data_mia_vision_3rec_1Oct22_ketem01oct81SmoothedNoRec_1Oct22_mat.csv
   [3] Original       D__data_mia_vision_3rec_1Oct22_ketem01oct81_1Oct22_mat.csv
   --> USING

In [13]:

# Robust compiler for 27 Oct 2022 trajectories
#
# Features
# --------
# • Automatically discovers filename variants
# • Priority:
#     S<bat> (smoothed) > Inbar<bat> > plain <bat>
# • Normalizes bat names by stripping leading "S" and "Inbar"
# • Ignores duplicate lower-priority copies
# • Works for both 50-column and 101-column csv files
# • Checks required columns
# • Removes empty trajectories
# • Detects duplicate trajectories
# • Writes master csv

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import re

DATA_DIR = Path("../csv_vision/csv_27Oct22")
OUTPUT_FILE = DATA_DIR / "master_acoustic_trajectory_data_27Oct22.csv"

REQUIRED = [
    "head1","head2","back",
    "Var28","Var29",
    "Var31","Var32",
    "Var34","Var35"
]

PATTERN = re.compile(
    r'_(?P<prefix>Inbar|S)?(?P<bat>[A-Za-z]+)27Oct(?P<traj>\d+)(?:_\d+)?_27Oct22Tal_mat\.csv$',
    re.IGNORECASE
)

def classify(prefix):
    if prefix is None:
        return 2,"Original"
    p=prefix.lower()
    if p=="s":
        return 0,"Smoothed"
    if p=="inbar":
        return 1,"Inbar"
    return 2,"Original"

def load_single_csv(path):
    try:
        df=pd.read_csv(path,header=[0,1])
    except Exception:
        df=pd.read_csv(path)

    if isinstance(df.columns,pd.MultiIndex):
        df.columns=df.columns.get_level_values(0)

    missing=[c for c in REQUIRED if c not in df.columns]
    if missing:
        print(f"Missing columns in {path.name}")
        print(missing)
        return None

    numeric_cols=["Frame_","Time"]+REQUIRED
    for c in numeric_cols:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c],errors="coerce")

    df["pos_x"]=df[["head1","head2","back"]].mean(axis=1)
    df["pos_y"]=df[["Var28","Var31","Var34"]].mean(axis=1)
    df["pos_z"]=df[["Var29","Var32","Var35"]].mean(axis=1)

    keep=[c for c in ["Frame_","Time","pos_x","pos_y","pos_z"] if c in df.columns]
    df=df[keep]
    df=df.dropna(subset=["pos_x","pos_y","pos_z"],how="all")
    if len(df)==0:
        return None
    return df

def main():
    files=sorted(DATA_DIR.glob("*.csv"))
    print(f"Found {len(files)} csv files")

    grouped=defaultdict(list)

    for f in files:
        m=PATTERN.search(f.name)
        if m is None:
            print("Could not parse:",f.name)
            continue

        bat=m.group("bat").lower()
        traj=int(m.group("traj"))
        priority,label=classify(m.group("prefix"))
        grouped[(bat,traj)].append((priority,label,f))

    merged=[]

    print("\n"+"="*80)
    print("FILE SELECTION")
    print("="*80)

    for key in sorted(grouped):
        candidates=sorted(grouped[key],key=lambda x:x[0])

        print(f"\n{key}")
        for p,l,f in candidates:
            print(f"   [{p}] {l:10s} {f.name}")

        chosen=candidates[0]
        print("   --> USING:",chosen[2].name)

        df=load_single_csv(chosen[2])
        if df is None:
            print("      Empty / invalid.")
            continue

        df["bat_name"]=key[0]
        df["trajectory_no"]=key[1]
        df["source_file"]=chosen[2].name
        merged.append(df)

    if not merged:
        raise RuntimeError("No valid trajectories found.")

    master=pd.concat(merged,ignore_index=True)
    cols=["bat_name","trajectory_no","Frame_","Time","pos_x","pos_y","pos_z","source_file"]
    master=master[[c for c in cols if c in master.columns]]
    master.to_csv(OUTPUT_FILE,index=False)

    print("\n"+"="*80)
    print("SUMMARY")
    print("="*80)
    print(master.groupby("bat_name")["trajectory_no"].nunique())

    print("\nChecking duplicate trajectories...")
    trajs={}
    for key,g in master.groupby(["bat_name","trajectory_no"]):
        trajs[key]=g[["pos_x","pos_y","pos_z"]].to_numpy()

    keys=list(trajs)
    dup=False
    for i in range(len(keys)):
        for j in range(i+1,len(keys)):
            A=trajs[keys[i]]
            B=trajs[keys[j]]
            if A.shape==B.shape and np.array_equal(A,B,equal_nan=True):
                print(keys[i],"<==>",keys[j])
                dup=True
    if not dup:
        print("No duplicate trajectories found.")

    print("\nSaved to")
    print(OUTPUT_FILE)

if __name__=="__main__":
    main()


Found 30 csv files

FILE SELECTION

('motek', 4)
   [0] Smoothed   D__data_mia_27Oct22Tal_Smotek27Oct4_392_27Oct22Tal_mat.csv
   [1] Inbar      D__data_mia_27Oct22Tal_Inbarmotek27Oct4_27Oct22Tal_mat.csv
   --> USING: D__data_mia_27Oct22Tal_Smotek27Oct4_392_27Oct22Tal_mat.csv

('motek', 5)
   [0] Smoothed   D__data_mia_27Oct22Tal_Smotek27Oct5_393_27Oct22Tal_mat.csv
   [1] Inbar      D__data_mia_27Oct22Tal_Inbarmotek27Oct5_27Oct22Tal_mat.csv
   --> USING: D__data_mia_27Oct22Tal_Smotek27Oct5_393_27Oct22Tal_mat.csv

('motek', 6)
   [0] Smoothed   D__data_mia_27Oct22Tal_Smotek27Oct6_394_27Oct22Tal_mat.csv
   [1] Inbar      D__data_mia_27Oct22Tal_Inbarmotek27Oct6_27Oct22Tal_mat.csv
   --> USING: D__data_mia_27Oct22Tal_Smotek27Oct6_394_27Oct22Tal_mat.csv

('motek', 7)
   [0] Smoothed   D__data_mia_27Oct22Tal_Smotek27Oct7_395_27Oct22Tal_mat.csv
   [1] Inbar      D__data_mia_27Oct22Tal_Inbarmotek27Oct7_27Oct22Tal_mat.csv
   --> USING: D__data_mia_27Oct22Tal_Smotek27Oct7_395_27Oct22Tal_mat.csv



In [14]:

# Robust compiler for 3 Nov 2022 trajectories
#
# Assumes all files are already the smoothed versions.
# Same pipeline as the robust 1 Oct compiler.

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------

DATA_DIR = Path("../csv_vision/csv_3nov22/smoothed")
OUTPUT_FILE = DATA_DIR / "master_vision_trajectory_data_3Nov22.csv"

REQUIRED = [
    "head1","head2","back",
    "Var28","Var29",
    "Var31","Var32",
    "Var34","Var35"
]

PATTERN = re.compile(
    r"S(?P<bat>[A-Za-z]+)3Nov22(?P<traj>\d+)",
    re.IGNORECASE
)

def load_single_csv(path):

    try:
        df = pd.read_csv(path, header=[0,1])
    except Exception:
        df = pd.read_csv(path)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    missing = [c for c in REQUIRED if c not in df.columns]

    if missing:
        print(f"\nMissing columns in {path.name}")
        print(missing)
        return None

    numeric_cols = ["Frame_", "Time"] + REQUIRED

    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["pos_x"] = df[["head1","head2","back"]].mean(axis=1)
    df["pos_y"] = df[["Var28","Var31","Var34"]].mean(axis=1)
    df["pos_z"] = df[["Var29","Var32","Var35"]].mean(axis=1)

    keep = [c for c in ["Frame_","Time","pos_x","pos_y","pos_z"] if c in df.columns]

    df = df[keep]

    df = df.dropna(
        subset=["pos_x","pos_y","pos_z"],
        how="all"
    )

    if len(df) == 0:
        return None

    return df


def main():

    files = sorted(DATA_DIR.glob("*.csv"))

    print(f"Found {len(files)} csv files")

    grouped = defaultdict(list)

    for f in files:

        m = PATTERN.search(f.name)

        if m is None:
            print("Could not parse:", f.name)
            continue

        bat = m.group("bat").lower()
        traj = int(m.group("traj"))

        grouped[(bat,traj)].append(f)

    merged=[]

    print("\n"+"="*80)
    print("FILE SELECTION")
    print("="*80)

    for key in sorted(grouped):

        if len(grouped[key])>1:
            print(f"WARNING: multiple files for {key}")

        chosen = sorted(grouped[key])[0]

        print(f"{key} -> {chosen.name}")

        df = load_single_csv(chosen)

        if df is None:
            print("   Empty / invalid.")
            continue

        df["bat_name"] = key[0]
        df["trajectory_no"] = key[1]
        df["source_file"] = chosen.name

        merged.append(df)

    if len(merged)==0:
        raise RuntimeError("No valid trajectories found.")

    master = pd.concat(
        merged,
        ignore_index=True
    )

    cols = [
        "bat_name",
        "trajectory_no",
        "Frame_",
        "Time",
        "pos_x",
        "pos_y",
        "pos_z",
        "source_file"
    ]

    master = master[[c for c in cols if c in master.columns]]

    master.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print("\n")
    print("="*80)
    print("SUMMARY")
    print("="*80)

    print(master.groupby("bat_name")["trajectory_no"].nunique())

    print("\nChecking duplicate trajectories...")

    trajs={}

    for key,g in master.groupby(["bat_name","trajectory_no"]):
        trajs[key]=g[["pos_x","pos_y","pos_z"]].to_numpy()

    keys=list(trajs)

    dup=False

    for i in range(len(keys)):
        for j in range(i+1,len(keys)):

            A=trajs[keys[i]]
            B=trajs[keys[j]]

            if A.shape!=B.shape:
                continue

            if np.array_equal(A,B,equal_nan=True):
                print(keys[i],"<==>",keys[j])
                dup=True

    if not dup:
        print("No duplicate trajectories found.")

    print("\nSaved to")
    print(OUTPUT_FILE)

if __name__=="__main__":
    main()


Found 52 csv files

FILE SELECTION
('ketem', 1) -> D__data_mia_vision_3rec_3Nov22Tal_SKetem3Nov221_3Nov22Tal_mat.csv
('ketem', 2) -> D__data_mia_vision_3rec_3Nov22Tal_Sketem3Nov222_3Nov22Tal_mat.csv
('ketem', 3) -> D__data_mia_vision_3rec_3Nov22Tal_Sketem3Nov223_3Nov22Tal_mat.csv
('ketem', 4) -> D__data_mia_vision_3rec_3Nov22Tal_Sketem3Nov224_3Nov22Tal_mat.csv
('ketem', 6) -> D__data_mia_vision_3rec_3Nov22Tal_Sketem3Nov226_3Nov22Tal_mat.csv
('ketem', 7) -> D__data_mia_vision_3rec_3Nov22Tal_SKetem3Nov227_3Nov22Tal_mat.csv
('ketem', 8) -> D__data_mia_vision_3rec_3Nov22Tal_SKetem3Nov228_3Nov22Tal_mat.csv
('ketem', 9) -> D__data_mia_vision_3rec_3Nov22Tal_SKetem3Nov229_3Nov22Tal_mat.csv
('ketem', 10) -> D__data_mia_vision_3rec_3Nov22Tal_Sketem3Nov2210_3Nov22Tal_mat.csv
('ketem', 12) -> D__data_mia_vision_3rec_3Nov22Tal_Sketem3Nov2212_3Nov22Tal_mat.csv
('ketem', 13) -> D__data_mia_vision_3rec_3Nov22Tal_SKetem3Nov2213_3Nov22Tal_mat.csv
('ketem', 14) -> D__data_mia_vision_3rec_3Nov22Tal_Sketem

In [16]:

# compile_6Oct22_master.py
#
# Robust compiler for 6 Oct 2022 trajectory files.
#
# Priority:
#   1. Inbar + Start (or Starting)
#   2. Inbar
#   3. Plain bat file
#
# Produces:
#   master_acoustic_trajectory_data_6Oct22.csv

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------

DATA_DIR = Path("../csv_vision/csv_6Oct22")
OUTPUT_FILE = DATA_DIR / "master_acoustic_trajectory_data_6Oct22.csv"

REQUIRED = [
    "head1","head2","back",
    "Var28","Var29",
    "Var31","Var32",
    "Var34","Var35"
]

PATTERN = re.compile(
    r'(?P<prefix>inbar)?(?P<bat>[A-Za-z]+)6oct(?P<traj>\d+)(?P<suffix>[A-Za-z]*)',
    re.IGNORECASE
)

def priority(name):
    n = name.lower()
    if "inbar" in n and "start" in n:
        return 0,"Inbar+Start"
    if "inbar" in n:
        return 1,"Inbar"
    return 2,"Plain"

def load_csv(path):
    try:
        df = pd.read_csv(path, header=[0,1])
    except Exception:
        df = pd.read_csv(path)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        print(f"[SKIP] Missing columns in {path.name}: {missing}")
        return None

    for c in ["Frame_","Time"] + REQUIRED:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["pos_x"] = df[["head1","head2","back"]].mean(axis=1)
    df["pos_y"] = df[["Var28","Var31","Var34"]].mean(axis=1)
    df["pos_z"] = df[["Var29","Var32","Var35"]].mean(axis=1)

    keep = [c for c in ["Frame_","Time","pos_x","pos_y","pos_z"] if c in df.columns]
    df = df[keep]

    df = df.dropna(subset=["pos_x","pos_y","pos_z"], how="all")
    if len(df)==0:
        return None

    return df

def main():

    grouped = defaultdict(list)

    for f in sorted(DATA_DIR.glob("*.csv")):
        m = PATTERN.search(f.stem)
        if not m:
            print("Couldn't parse:", f.name)
            continue

        bat = m.group("bat").lower()
        traj = int(m.group("traj"))
        p,label = priority(f.name)
        grouped[(bat,traj)].append((p,label,f))

    merged=[]

    print("="*80)
    print("FILE SELECTION")
    print("="*80)

    for key in sorted(grouped):
        bat,traj = key
        candidates = sorted(grouped[key], key=lambda x:x[0])

        print(f"\n{bat}  Trajectory {traj}")
        for p,l,f in candidates:
            print(f"   [{p}] {l:12s} {f.name}")

        chosen = candidates[0]
        print("   --> USING:", chosen[2].name)

        df = load_csv(chosen[2])
        if df is None:
            print("      Empty / invalid.")
            continue

        df["bat_name"]=bat
        df["trajectory_no"]=traj
        df["source_file"]=chosen[2].name

        merged.append(df)

    if not merged:
        raise RuntimeError("No valid trajectories found.")

    master = pd.concat(merged, ignore_index=True)

    cols = ["bat_name","trajectory_no","Frame_","Time",
            "pos_x","pos_y","pos_z","source_file"]
    cols=[c for c in cols if c in master.columns]
    master=master[cols]

    master.to_csv(OUTPUT_FILE,index=False)

    print("\n"+"="*80)
    print("SUMMARY")
    print("="*80)
    print(master.groupby("bat_name")["trajectory_no"].nunique())

    print("\nChecking duplicate trajectories...")

    trajs={}
    for key,g in master.groupby(["bat_name","trajectory_no"]):
        trajs[key]=g[["pos_x","pos_y","pos_z"]].to_numpy()

    keys=list(trajs)
    dup=False

    for i in range(len(keys)):
        for j in range(i+1,len(keys)):
            A=trajs[keys[i]]
            B=trajs[keys[j]]
            if A.shape!=B.shape:
                continue
            if np.array_equal(A,B,equal_nan=True):
                print(keys[i],"<==>",keys[j])
                dup=True

    if not dup:
        print("No duplicate trajectories found.")

    print("\nSaved to")
    print(OUTPUT_FILE)

if __name__=="__main__":
    main()


FILE SELECTION

motek  Trajectory 1
   [0] Inbar+Start  D__data_mia_Oct622Tal_Inbarmotek6Oct1Start_Oct622Tal_mat.csv
   [1] Inbar        D__data_mia_Oct622Tal_Inbarmotek6Oct1_Oct622Tal_mat.csv
   --> USING: D__data_mia_Oct622Tal_Inbarmotek6Oct1Start_Oct622Tal_mat.csv

motek  Trajectory 5
   [1] Inbar        D__data_mia_Oct622Tal_Inbarmotek6Oct5_Oct622Tal_mat.csv
   --> USING: D__data_mia_Oct622Tal_Inbarmotek6Oct5_Oct622Tal_mat.csv

motek  Trajectory 8
   [0] Inbar+Start  D__data_mia_Oct622Tal_Inbarmotek6Oct8Start_Oct622Tal_mat.csv
   [1] Inbar        D__data_mia_Oct622Tal_Inbarmotek6Oct8_Oct622Tal_mat.csv
   --> USING: D__data_mia_Oct622Tal_Inbarmotek6Oct8Start_Oct622Tal_mat.csv

motek  Trajectory 13
   [2] Plain        D__data_mia_Oct622Tal_motek6Oct13_Oct622Tal_mat.csv
   --> USING: D__data_mia_Oct622Tal_motek6Oct13_Oct622Tal_mat.csv
      Empty / invalid.

motek  Trajectory 27
   [1] Inbar        D__data_mia_Oct622Tal_Inbarmotek6Oct27_Oct622Tal_mat.csv
   --> USING: D__data_mia_Oct6

## 2. Trajectory Smoothing (Savgol Filter)

Smooths `pos_x`, `pos_y`, `pos_z` in each master file and writes a companion `_smoothed.csv` file.

In [1]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
import os

# ============================================================
# 1. Configuration
# ============================================================
MASTER_FILES = {
    "30.10.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_30Oct.csv",
    "24.12.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_24Dec.csv",
    "26.12.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_26Dec.csv",
    "31.12.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_31Dec.csv",
    "03.11.22": "../masters_csv/master_vision_trajectory_data_3Nov22.csv",
    "01.10.22": "../masters_csv/master_vision_trajectory_data_1Oct22.csv",
    "06.10.22": "../masters_csv/master_vision_trajectory_data_6Oct22.csv",
    "27.10.22": "../masters_csv/master_vision_trajectory_data_27Oct22.csv"
}

WINDOW_LENGTH = 15   # odd, > polyorder
POLYORDER = 7

# ============================================================
# 2. Process each file safely
# ============================================================
for date_key, input_path in MASTER_FILES.items():
    print(f"\nProcessing {date_key}: {input_path}")

    df = pd.read_csv(input_path)

    # Ensure group keys are consistently string
    df["bat_name"] = df["bat_name"].astype(str)
    df["trajectory_no"] = df["trajectory_no"].astype(str)

    # Record original row order
    df["__row_order__"] = np.arange(len(df))

    # List to collect all modified groups
    chunks = []

    # Iterate through each trajectory explicitly
    for (bat, traj), group in df.groupby(["bat_name", "trajectory_no"], sort=False):
        group = group.copy()   # work on a copy

        n = len(group)
        if n >= WINDOW_LENGTH:
            for col in ["pos_x", "pos_y", "pos_z"]:
                group[col] = savgol_filter(group[col].values, WINDOW_LENGTH, POLYORDER)
        else:
            print(f"Warning: {bat} traj {traj} has only {n} points; not smoothed.")

        chunks.append(group)

    # Recombine and sort back to original order
    smoothed = pd.concat(chunks, ignore_index=False)
    smoothed = smoothed.sort_values("__row_order__")
    smoothed = smoothed.drop(columns="__row_order__")

    # Save to file with "_smoothed" inserted before the extension
    dir_name, file_name = os.path.split(input_path)
    name, ext = os.path.splitext(file_name)
    output_path = os.path.join(dir_name, f"{name}_smoothed{ext}")

    smoothed.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

    # ======= Optional verification =======
    test = pd.read_csv(output_path)
    missing = [col for col in ["bat_name", "trajectory_no"] if col not in test.columns]
    if missing:
        print(f"ERROR: Columns missing in output: {missing}")
    else:
        print(f"Verified: 'bat_name' and 'trajectory_no' columns present. Rows: {len(test)}")

print("\nAll files processed.")


Processing 30.10.24: ../csv_acoustic/masters/master_acoustic_trajectory_data_30Oct.csv
Saved: ../csv_acoustic/masters\master_acoustic_trajectory_data_30Oct_smoothed.csv
Verified: 'bat_name' and 'trajectory_no' columns present. Rows: 36220

Processing 24.12.24: ../csv_acoustic/masters/master_acoustic_trajectory_data_24Dec.csv
Saved: ../csv_acoustic/masters\master_acoustic_trajectory_data_24Dec_smoothed.csv
Verified: 'bat_name' and 'trajectory_no' columns present. Rows: 16000

Processing 26.12.24: ../csv_acoustic/masters/master_acoustic_trajectory_data_26Dec.csv
Saved: ../csv_acoustic/masters\master_acoustic_trajectory_data_26Dec_smoothed.csv
Verified: 'bat_name' and 'trajectory_no' columns present. Rows: 44396

Processing 31.12.24: ../csv_acoustic/masters/master_acoustic_trajectory_data_31Dec.csv
Saved: ../csv_acoustic/masters\master_acoustic_trajectory_data_31Dec_smoothed.csv
Verified: 'bat_name' and 'trajectory_no' columns present. Rows: 68336

Processing 03.11.22: ../masters_csv/mas

In [1]:
from ipynb.fs.full.data_analysis_clean_dual import find_bifurcations

ModuleNotFoundError: No module named 'ipynb'